# Edge Generator: ZINC Molecule Generation

Fit an edge-growth model, remove edges from one molecule, then regrow it.

In [ ]:
from pathlib import Path
import runpy

roots = (Path.cwd(), *Path.cwd().parents)
candidates = [
    root / relative
    for root in roots
    for relative in (
        "notebooks/_bootstrap.py",
        "repos/abstractgraph-generative/notebooks/_bootstrap.py",
    )
]
bootstrap_path = next((path for path in candidates if path.is_file()), None)
if bootstrap_path is None:
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")
runpy.run_path(str(bootstrap_path))

In [ ]:
from typing import cast

from sklearn.ensemble import RandomForestClassifier  # type: ignore[reportMissingModuleSource]

from abstractgraph.graphs import AbstractGraph
from abstractgraph.operators import neighborhood
from abstractgraph.vectorize import AbstractGraphTransformer
from nsppk import NSPPK
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_ml.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
)
from abstractgraph_generative.edge_generator import EdgeGenerator, remove_edges

In [ ]:
graphs, _ = ZINCLoader(on_error="skip").load(
    "zinc_250k",
    limit=500,
    min_node_count=10,
    max_node_count=13,
)
print(f"Loaded {len(graphs)} molecules")

In [ ]:
def one_hop_neighborhood(abstract_graph: AbstractGraph) -> AbstractGraph:
    return cast(AbstractGraph, neighborhood(abstract_graph, radius=1))


feasibility = FeasibilityEstimator([
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=one_hop_neighborhood,
        nbits=14,
        parallel=False,
        n_jobs=1,
    )
])
vectorizer = cast(
    AbstractGraphTransformer,
    NSPPK(radius=1, distance=4, connector=1, nbits=14, dense=True),
)
scorer = GraphEstimator(
    transformer=vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced_subsample",
        random_state=0,
        n_jobs=1,
    ),
)

generator = EdgeGenerator(
    feasibility_estimator=feasibility,
    graph_estimator=scorer,
    n_negative_per_positive=1,
    n_replicates=1,
    beam_size=3,
    max_restarts=1,
    fit_n_jobs=1,
    seed=0,
).fit(graphs)

In [ ]:
source = graphs[0]
start, target_edges = remove_edges(source, size=0.6, seed=0)
path = generator.generate(start, n_edges=target_edges, return_path=True)
if path:
    print(f"Generated {len(path) - 1} edge additions")
    draw_molecules([source, start, *path], n_graphs_per_line=4)
else:
    print("No valid graph generated; try a larger training slice.")